# 06 — Feature Engineering

## Purpose
This notebook transforms the clean sales data into **ML-ready datasets** — one CSV file
per branch, each containing daily-level rows with all the features the forecasting model needs.

This is the most important step before modeling. Every decision here (which products to keep,
how to compute rolling windows, how to aggregate) directly impacts model quality.

## What we do
1. Load and clean the data (same steps as previous notebooks)
2. Create holiday columns
3. Filter to the top 83 products per branch (90% cumulative volume rule)
4. Remove non-commercial items
5. Keep only completed sales (filter out cancellations and voids)
6. Drop columns not needed for ML
7. Aggregate to daily level: 1 row per (branch, date, product)
8. Compute rolling window features: qty sold in past 1/3/7/15/30/60/90/180/365 days
9. Export one CSV per branch to `data/processed/`

## Input
`data/intermediate/datanomodifier.csv`

## Output
7 files in `data/processed/`: `branch_carreta.csv`, `branch_credi_club.csv`, etc.

## Run order
Run after `01_data_cleaning.ipynb`. This notebook MUST run before `07_top_products.ipynb`.

In [ ]:
import os

# ─── PATH CONFIGURATION ───────────────────────────────────────────────
# Option A — After cloning the repo (default, USE_GITHUB = False)
#   git clone https://github.com/DiegoLarrieta/PanemReto
#   cd PanemReto/notebooks
#   jupyter notebook
#   Paths resolve automatically — no changes needed.
#
# Option B — Read directly from GitHub, e.g. Google Colab (USE_GITHUB = True)
#   Works for notebooks 07 (processed branch CSVs are on GitHub).
#   Notebooks 00-06 need the intermediate files which are NOT in the repo
#   (too large). Run 00_data_pipeline.ipynb locally first to generate them.
# ──────────────────────────────────────────────────────────────────────────

USE_GITHUB   = False
GITHUB_BASE  = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR    = f"{GITHUB_BASE}/data/processed"
    WEATHER_DIR      = f"{GITHUB_BASE}/data/weather"
    RAW_DIR          = f"{GITHUB_BASE}/data/raw/Complete Data"
    INTERMEDIATE_DIR = None  # not in repo — generate locally with 00_data_pipeline.ipynb
else:
    PROCESSED_DIR    = PROCESSED_DIR
    WEATHER_DIR      = WEATHER_DIR
    RAW_DIR          = RAW_DIR
    INTERMEDIATE_DIR = INTERMEDIATE_DIR


## Step 1 — Imports and paths

## Step 2 — Load and clean data

Same cleaning block as previous notebooks. We re-apply it here so this notebook
can run independently.

In [ ]:
df = pd.read_csv(os.path.join(INTERMEDIATE_DIR, "datanomodifier.csv"), low_memory=False)

for col in ["operating_date", "closing_time", "captured_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df["is_modifier"] = (
    df["is_modifier"].astype("string").str.strip().str.lower()
    .map({"true": True, "false": False}).fillna(False).astype(bool)
)
df["quantity"]     = pd.to_numeric(df["quantity"],     errors="coerce").fillna(0)
df["total_ticket"] = pd.to_numeric(df["total_ticket"], errors="coerce").fillna(0)
df["tavg"]         = pd.to_numeric(df["tavg"],         errors="coerce")

df = df[~df["group"].isin(["CAFE Y BEBIDAS CALIENTES", "JUGOS Y BEBIDAS FRIAS"])].copy()
df["sucursal"] = df["sucursal"].replace({
    "Panem - Hotel Kavia N"      : "Panem - Hotel Kavia",
    "Panem - Plaza QIN N"        : "Panem - Plaza QIN",
    "Panem - Hospital Zambrano N": "Panem - Hospital Zambrano",
    "Panem - La Carreta N"       : "Panem - Carreta",
})
df["item"] = df["item"].replace({"SANDWITCH ENSALADA POLLO": "SANDWICH ENSALADA POLLO"})
df["cold_or_warm"] = np.where(df["tavg"] >= 25, "warm", "cold")

print(f"Loaded {len(df):,} rows | {df['sucursal'].nunique()} branches")

## Step 3 — Add holiday columns

Holiday flags are added here (before aggregation) so they survive the groupby step.
Each date is classified as 'Festivo Oficial', 'Puente', or 'No holiday'.

In [ ]:
holiday_map = {
    "2022-01-01": "Festivo Oficial", "2022-02-07": "Puente", "2022-03-21": "Puente",
    "2022-05-01": "Festivo Oficial", "2022-09-16": "Puente", "2022-11-21": "Puente", "2022-12-25": "Festivo Oficial",
    "2023-01-01": "Festivo Oficial", "2023-02-06": "Puente", "2023-03-20": "Puente",
    "2023-05-01": "Puente", "2023-09-16": "Festivo Oficial", "2023-11-20": "Puente", "2023-12-25": "Puente",
    "2024-01-01": "Puente", "2024-02-05": "Puente", "2024-03-18": "Puente",
    "2024-05-01": "Festivo Oficial", "2024-06-02": "Festivo Oficial", "2024-09-16": "Puente",
    "2024-10-01": "Festivo Oficial", "2024-11-18": "Puente", "2024-12-25": "Festivo Oficial",
    "2025-01-01": "Festivo Oficial", "2025-02-03": "Puente", "2025-03-17": "Puente",
    "2025-05-01": "Festivo Oficial", "2025-09-16": "Festivo Oficial", "2025-11-17": "Puente", "2025-12-25": "Festivo Oficial",
    "2026-01-01": "Festivo Oficial", "2026-02-02": "Puente", "2026-03-16": "Puente",
    "2026-05-01": "Puente", "2026-09-16": "Festivo Oficial", "2026-11-16": "Puente", "2026-12-25": "Puente",
}
holiday_dt_map = {pd.to_datetime(k): v for k, v in holiday_map.items()}

df["operating_date"] = df["operating_date"].dt.normalize()
df["holiday_type"]   = df["operating_date"].map(holiday_dt_map).fillna("No holiday")
df["holidays"]       = df["holiday_type"] != "No holiday"

print("Holiday type counts:")
print(df["holiday_type"].value_counts())

## Step 4 — Filter to top 83 products (90% volume rule)

**Why 90%?**
The sales distribution follows a power law: a small number of products account for the
vast majority of volume, and hundreds of products sell very rarely.

Modeling rare products is difficult (too few data points) and low-value for stock planning.
We keep only the products that together account for 90% of total volume — which turns out
to be the top 83 products.

Everything else is noise for our forecasting purpose.

In [ ]:
# Calculate cumulative volume share per product
prod_volume = (
    df.groupby("item")["quantity"].sum()
      .sort_values(ascending=False)
)
cumulative_pct = prod_volume.cumsum() / prod_volume.sum()

# Find the cutoff: first index where cumulative % >= 90%
cutoff_idx = (cumulative_pct >= 0.90).idxmax()
cutoff_pos = prod_volume.index.get_loc(cutoff_idx) + 1
print(f"Products needed to reach 90% volume: {cutoff_pos}")

top_products = prod_volume.head(83).index.tolist()

rows_before = len(df)
df = df[df["item"].isin(top_products)].copy()
print(f"Rows before filter: {rows_before:,}")
print(f"Rows after filter:  {len(df):,}")
print(f"Unique products:    {df['item'].nunique()}")

## Step 5 — Remove non-commercial items

- **SUBSIDIO TEC**: An employee subsidy item — it represents internal discounts, not actual
  product demand. Including it would corrupt forecasting.
- **PAN DE MUERTO**: A seasonal product sold only around Día de Muertos (late October / early November).
  Its extreme seasonality makes it an outlier that could distort the model for other products.

In [ ]:
items_to_remove = ["SUBSIDIO TEC", "PAN DE MUERTO"]

rows_before = len(df)
df = df[~df["item"].isin(items_to_remove)].copy()
print(f"Removed {rows_before - len(df):,} rows for {items_to_remove}")
print(f"Unique products remaining: {df['item'].nunique()}")

## Step 6 — Keep only completed sales

The raw POS data contains multiple action types: `Venta` (sale), cancellations, voids,
and courtesy comps. We only want to predict actual demand — completed sales.

Filtering to `action == 'Venta'` removes all reversals.

In [ ]:
rows_before = len(df)
df = df[df["action"] == "Venta"].copy()
print(f"Rows after keeping only Venta: {len(df):,}  (removed {rows_before - len(df):,} non-sale rows)")

## Step 7 — Drop columns not needed for ML

The raw data has 47 columns but the model only needs a small subset. We drop everything
that is either:
- A transaction identifier (POS IDs, order IDs)
- A financial metric we're not predicting (prices, taxes, discounts)
- An operational detail (server name, terminal ID)
- Redundant with columns we're keeping

The columns we keep: `sucursal`, `operating_date`, `item`, `quantity`, `day_name`,
`week_number`, `tavg`, `cold_or_warm`, `holiday_type`, `holidays`, `month`

In [ ]:
cols_to_drop = [
    "closing_time", "captured_time", "pdv_txn_id", "order_id", "order_type",
    "order_subtype", "table_number", "party_size", "server",
    "terminal", "capture_terminal", "action", "modifier",
    "description", "is_modifier", "unit_price_with_mods", "unit_price",
    "cost_actual", "cost_with_mods", "cost_ideal", "discount",
    "subtotal_ticket", "iva_ticket", "ieps_ticket", "total_ticket",
    "subtotal_item", "iva_item", "ieps_item", "total_item",
    "subtotal_cortesia_cancel", "iva_cortesia_cancel", "ieps_cortesia_cancel", "total_cortesia_cancel",
    "subtotal_anulacion", "iva_anulacion", "ieps_anulacion", "total_anulacion",
    "clave_platillo", "group_type", "group", "day_part", "hour",
    "_source_file"
]
actual_drops = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=actual_drops)

print(f"Remaining columns ({len(df.columns)}):")
print(df.columns.tolist())

## Step 8 — Aggregate to daily level

Right now each row is one order line item. We want **one row per (branch, date, product)**.

We `groupby` on those three keys and:
- **Sum** `quantity` — total units sold that day
- **Take the first** value for all other columns — they're the same for all rows
  with the same (branch, date, product) combination

In [ ]:
first_cols = [c for c in df.columns if c not in ["sucursal", "operating_date", "item", "quantity"]]

agg_dict = {"quantity": "sum"}
agg_dict.update({c: "first" for c in first_cols})

df = df.groupby(["sucursal", "operating_date", "item"], as_index=False).agg(agg_dict)

print(f"Rows after daily aggregation: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.sample(5)

## Step 9 — Rolling window features

This is the most important feature engineering step.

**What is a rolling window feature?**
For each product at each branch, we calculate: 'how many units were sold in the past N days?'
These become the model's lagged features — the model learns patterns like:
'if this product sold a lot in the last 7 days, it will probably sell well tomorrow too.'

**Why 9 different windows (1, 3, 7, 15, 30, 60, 90, 180, 365 days)?**
Different products have different demand cycles:
- 1-day lag catches very recent momentum
- 7-day captures weekly seasonality
- 30-day captures monthly patterns
- 365-day captures year-over-year seasonality

**Critical detail — handling sparse products:**
Many products don't sell every single day. If we compute rolling windows only on
days when there WAS a sale, we would miss zeros.

Solution: we reindex each (branch, product) time series to have a row for EVERY calendar day
(filling missing days with quantity=0), compute the rolling windows, then join the results
back to the original rows only.

**Why `shift(1)`?**
The rolling window is shifted by 1 day so that today's sale is NOT included in today's features.
Without the shift we would be 'leaking' the target variable into the features — the model
would know the answer before making a prediction, which is cheating.

In [ ]:
df["operating_date"] = pd.to_datetime(df["operating_date"])
df = df.sort_values(["sucursal", "item", "operating_date"]).reset_index(drop=True)

# Define rolling window sizes in days
windows = {
    "qty_roll_1"  : 1,
    "qty_roll_3"  : 3,
    "qty_roll_7"  : 7,
    "qty_roll_15" : 15,
    "qty_roll_30" : 30,
    "qty_roll_60" : 60,
    "qty_roll_90" : 90,
    "qty_roll_180": 180,
    "qty_roll_365": 365,
}

groups = []
for (branch, item), grp in df.groupby(["sucursal", "item"]):
    # Reindex to full date range — inserts 0-quantity rows for missing days
    grp = grp.set_index("operating_date")
    full_range = pd.date_range(grp.index.min(), grp.index.max(), freq="D")
    grp = grp.reindex(full_range)
    grp["quantity"] = grp["quantity"].fillna(0)

    # Compute each rolling window (shift(1) excludes current day)
    for col_name, w in windows.items():
        grp[col_name] = grp["quantity"].shift(1).rolling(window=w, min_periods=1).sum().fillna(0)

    grp["sucursal"] = branch
    grp["item"]     = item
    groups.append(grp)

df_rolled = pd.concat(groups).reset_index().rename(columns={"index": "operating_date"})

roll_cols = list(windows.keys())

# Merge rolling features back — only for actual sale rows (not the gap-filling zeros)
df = df.merge(
    df_rolled[["sucursal", "item", "operating_date"] + roll_cols],
    on=["sucursal", "item", "operating_date"],
    how="left"
)

print(f"Final rows: {len(df):,}")
print(f"Final columns ({len(df.columns)}): {df.columns.tolist()}")
df.head(5)

## Step 10 — Export one CSV per branch

We split the dataframe by branch and save one file per branch to `data/processed/`.

Each file is self-contained: it has all the rows and features for one branch, ready
for a model to train on without needing any other files.

In [ ]:
branch_name_map = {
    "Panem - Carreta"           : "branch_carreta.csv",
    "Panem - Credi Club"        : "branch_credi_club.csv",
    "Panem - Hospital Zambrano" : "branch_hospital_zambrano.csv",
    "Panem - Hotel Kavia"       : "branch_hotel_kavia.csv",
    "Panem - Plaza Nativa"      : "branch_plaza_nativa.csv",
    "Panem - Plaza QIN"         : "branch_plaza_qin.csv",
    "Panem - Punto Valle"       : "branch_punto_valle.csv",
}

for branch, filename in branch_name_map.items():
    branch_df = df[df["sucursal"] == branch].copy()
    path = os.path.join(PROCESSED_DIR, filename)
    branch_df.to_csv(path, index=False)
    print(f"{branch:<35} → {len(branch_df):>7,} rows  saved to {filename}")

print(f"\nDone. {len(branch_name_map)} branch files saved to data/processed/")

## Summary

After running this notebook, `data/processed/` contains 7 CSV files, one per branch.
Each file has 20 columns:

| Column | Description |
|--------|-------------|
| `sucursal` | Branch name |
| `operating_date` | Sale date |
| `item` | Product name |
| `quantity` | **Target variable** — units sold that day |
| `day_name` | Day of week (Spanish) |
| `week_number` | ISO week |
| `tavg` | Average temperature °C |
| `cold_or_warm` | 'warm' or 'cold' |
| `holiday_type` | Holiday classification |
| `holidays` | Boolean holiday flag |
| `month` | Month number |
| `qty_roll_1..365` | Rolling window features (9 columns) |

**Next step:** `07_top_products.ipynb` — identify the top 5 products per branch and prepare final model-ready data.